# Text Retrieval: Document Vectors, Similarity, and Retrieval Effectiveness

**Companion to slides 20–26** of `Info_retrieval_01.pptx`. This follows the term-weighting notebook. The equations and numerical examples below come from the lecture; executable Python cells verify them. Additional illustrative exercises are labeled separately.

## 1. Queries and documents as vectors (slide 20)

For a collection containing $n$ distinct terms, represent a document and a query as:

$$\mathbf d=(w_1,w_2,\ldots,w_n), \qquad \mathbf q=(q_1,q_2,\ldots,q_n)$$

Here $w_i$ is the weight of term $i$ in the document. The lecture uses the **term-frequency weight** for $q_i$ when term $i$ occurs in the query, and $q_i=0$ otherwise.

The term order must be the same in both vectors.

### Worked example: calculate TF-IDF document vectors, then match a query

The earlier term-weighting lesson showed how to assign weights to document terms. Now we use those weights to **rank documents for a user's query**.

Consider three documents:

- **D1:** "data mining techniques"
- **D2:** "machine learning techniques"
- **D3:** "data mining applications"

Use the same vocabulary order for all vectors:

$$V=(\text{data},\ \text{mining},\ \text{techniques},\ \text{machine},\ \text{learning},\ \text{applications})$$

#### Step 1: Calculate document frequency (DF) and inverse document frequency (IDF)

For this worked example, use **raw term frequency** and **base-10 IDF**:

$$\operatorname{TF}(t,d)=\text{number of occurrences of }t\text{ in }d$$

$$\operatorname{IDF}(t)=\log_{10}\left(\frac{N}{\operatorname{DF}(t)}\right),\qquad N=3$$

$$\operatorname{TFIDF}(t,d)=\operatorname{TF}(t,d)\times\operatorname{IDF}(t)$$

Each term occurs at most once per document, so TF is either 0 or 1. Terms appearing in **two documents** have:

$$\log_{10}(3/2)\approx0.1761$$

Terms appearing in **one document** have:

$$\log_{10}(3/1)\approx0.4771$$

| Term | DF | IDF |
|:--|--:|--:|
| data | 2 | 0.1761 |
| mining | 2 | 0.1761 |
| techniques | 2 | 0.1761 |
| machine | 1 | 0.4771 |
| learning | 1 | 0.4771 |
| applications | 1 | 0.4771 |

#### Step 2: Construct the document vectors

| Document | data | mining | techniques | machine | learning | applications |
|:--|--:|--:|--:|--:|--:|--:|
| D1 | 0.1761 | 0.1761 | 0.1761 | 0 | 0 | 0 |
| D2 | 0 | 0 | 0.1761 | 0.4771 | 0.4771 | 0 |
| D3 | 0.1761 | 0.1761 | 0 | 0 | 0 | 0.4771 |

For example,

$$\mathbf d_1=(0.1761,\ 0.1761,\ 0.1761,\ 0,\ 0,\ 0)$$

These are **calculated TF-IDF weights**, not arbitrarily assigned values.

#### Step 3: Construct a query vector

Suppose the user searches for **"data mining"**. For clarity, we use **binary query weights** (1 if a term occurs, 0 otherwise):

$$\mathbf q=(1,\ 1,\ 0,\ 0,\ 0,\ 0)$$

The lecture slide specifies **TF weights for queries**, which is a valid alternative. Here, each query term occurs once, so both raw TF and normalized augmented TF also assign 1 to each present term.

#### Step 4: Compare the query with each document

Use dot-product similarity:

$$\operatorname{sim}(\mathbf q,\mathbf d)=\sum_{i=1}^{6}q_iw_i$$

$$\operatorname{sim}(\mathbf q,\mathbf d_1)=1(0.1761)+1(0.1761)\approx0.3522$$

$$\operatorname{sim}(\mathbf q,\mathbf d_2)=0$$

$$\operatorname{sim}(\mathbf q,\mathbf d_3)\approx0.3522$$

**Ranking:** D1 and D3 tie for first; D2 is third. Only weights of terms appearing in the query contribute to the dot product.

**Key distinction:** TF-IDF determines document term weights; the query vector allows us to compare a search request with each document in the same term space.

In [ ]:
# Calculate TF-IDF from the actual three-document collection
import math
from collections import Counter

documents_example = {
    "D1": "data mining techniques",
    "D2": "machine learning techniques",
    "D3": "data mining applications",
}
vocabulary_example = [
    "data", "mining", "techniques", "machine", "learning", "applications"
]
N_example = len(documents_example)
token_counts_example = {
    name: Counter(text.split()) for name, text in documents_example.items()
}

df_example = {
    term: sum(counts[term] > 0 for counts in token_counts_example.values())
    for term in vocabulary_example
}
idf_example = {
    term: math.log10(N_example / df_example[term])
    for term in vocabulary_example
}

document_vectors_example = {
    name: [counts[term] * idf_example[term] for term in vocabulary_example]
    for name, counts in token_counts_example.items()
}

print("Term          DF    IDF")
for term in vocabulary_example:
    print(f"{term:13s} {df_example[term]:2d}   {idf_example[term]:.4f}")

print("\nDocument TF-IDF vectors:")
for name, weights in document_vectors_example.items():
    print(name, [round(w, 4) for w in weights])

# Binary query weights (also equal to raw TF for this particular query)
query_text_example = "data mining"
query_terms_example = set(query_text_example.split())
query_vector_example = [
    1 if term in query_terms_example else 0 for term in vocabulary_example
]

print("\nQuery vector:", query_vector_example)
print("Dot-product similarities:")
for name, weights in document_vectors_example.items():
    similarity = sum(q * w for q, w in zip(query_vector_example, weights))
    print(f"{name}: {similarity:.4f}")


In [ ]:
vocabulary = ["data", "mining", "web", "world"]
document_weights = {"data": 0.2, "mining": 0.0, "web": 0.3, "world": 1.0}
query_weights = {"data": 0.75, "mining": 0.75, "web": 0.0, "world": 1.0}
d = [document_weights.get(term, 0.0) for term in vocabulary]
q = [query_weights.get(term, 0.0) for term in vocabulary]
print("Terms:   ", vocabulary)
print("Document:", d)
print("Query:   ", q)

## 2. Dot-product similarity (slides 21–22)

$$\operatorname{sim}(\mathbf q,\mathbf d)=\mathbf q\cdot\mathbf d=\sum_{i=1}^{n}q_iw_i$$

The lecture's example uses:

$$\mathbf d=(0.2,0,0.3,1),\qquad \mathbf q=(0.75,0.75,0,1)$$

$$\operatorname{sim}(\mathbf q,\mathbf d)=(0.75)(0.2)+(0.75)(0)+(0)(0.3)+(1)(1)=\boxed{1.15}$$

**Observations from slide 22:** Shared terms contribute to the score, higher weights contribute more, and longer documents may be favored. Dot-product similarity has no fixed upper bound.

In [ ]:
def dot_product(a, b):
    if len(a) != len(b):
        raise ValueError("Vectors must have the same length")
    return sum(x * y for x, y in zip(a, b))

contributions = [x * y for x, y in zip(q, d)]
print("Term contributions:")
for term, value in zip(vocabulary, contributions):
    print(f"  {term:8s} {value:.2f}")
print(f"Dot product = {dot_product(q, d):.2f}")

## 3. Cosine similarity (slides 23–24)

Cosine similarity normalizes the dot product by the lengths of **both** vectors:

$$\operatorname{sim}(\mathbf q,\mathbf d)=\cos\theta=\frac{\mathbf q\cdot\mathbf d}{\|\mathbf q\|\,\|\mathbf d\|}$$

where the Euclidean norm is:

$$\|\mathbf d\|=\sqrt{\sum_{i=1}^{n}w_i^2},\qquad \|\mathbf q\|=\sqrt{\sum_{i=1}^{n}q_i^2}$$

For the lecture's vectors:

$$\|\mathbf d\|=\sqrt{0.2^2+0^2+0.3^2+1^2}=\sqrt{1.13}\approx1.063$$

$$\|\mathbf q\|=\sqrt{0.75^2+0.75^2+0^2+1^2}=\sqrt{2.125}\approx1.458$$

$$\cos\theta=\frac{1.15}{(1.063)(1.458)}\approx\boxed{0.742}$$

For nonnegative term weights, cosine similarity lies between 0 and 1. Unlike raw dot product, it accounts for vector length.

In [ ]:
import math

def norm(a):
    return math.sqrt(sum(x * x for x in a))

def cosine_similarity(a, b):
    denominator = norm(a) * norm(b)
    return dot_product(a, b) / denominator if denominator else 0.0

print(f"||d|| = {norm(d):.3f}")
print(f"||q|| = {norm(q):.3f}")
print(f"Cosine similarity = {cosine_similarity(q, d):.3f}")

### Additional illustration: effect of document length

The following **new example** demonstrates why cosine normalization can be useful. Compare $\mathbf d_1=(1,1)$ and $\mathbf d_2=(10,10)$ against $\mathbf q=(1,1)$. They point in the same direction, so their cosine similarities are equal even though their dot products differ.

In [ ]:
query = [1, 1]
for name, doc in [("d1", [1, 1]), ("d2", [10, 10])]:
    print(f"{name}: dot={dot_product(query, doc):.1f}, cosine={cosine_similarity(query, doc):.3f}")

## 4. Retrieval effectiveness: recall and precision (slides 25–26)

The lecture defines a relevant document as one that the user judges useful for the query. Relevance is subjective.

$$\operatorname{Recall}=\frac{\text{number of retrieved relevant documents}}{\text{total number of relevant documents}}$$

$$\operatorname{Precision}=\frac{\text{number of retrieved relevant documents}}{\text{total number of retrieved documents}}$$

**Slide 26 example:** There are 20 relevant documents in the collection. The system retrieves 10 documents, of which 6 are relevant.

$$\operatorname{Recall}=\frac{6}{20}=\boxed{0.30}$$

$$\operatorname{Precision}=\frac{6}{10}=\boxed{0.60}$$

The ideal is recall = 1 and precision = 1, but this is difficult to achieve in practice. The lecture notes that increasing recall often reduces precision.

In [ ]:
def recall_precision(retrieved_relevant, total_relevant, total_retrieved):
    recall = retrieved_relevant / total_relevant if total_relevant else 0.0
    precision = retrieved_relevant / total_retrieved if total_retrieved else 0.0
    return recall, precision

recall, precision = recall_precision(6, 20, 10)
print(f"Recall:    {recall:.2f} ({recall:.0%})")
print(f"Precision: {precision:.2f} ({precision:.0%})")

### Additional illustration: precision–recall as more results are retrieved

This **new example** uses an illustrative ranked result list. `True` means a retrieved document is relevant. We assume there are 6 relevant documents in the full collection.

In [ ]:
ranked_results = [True, True, False, True, False, False, True, False, True, False]
total_relevant = 6
found = 0
print("Top-k   Recall   Precision")
for k, relevant in enumerate(ranked_results, start=1):
    found += int(relevant)
    r, p = recall_precision(found, total_relevant, k)
    print(f"{k:>5}   {r:>6.2f}   {p:>9.2f}")

## 5. Practice questions

1. For $\mathbf q=(1,0,1)$ and $\mathbf d=(2,1,2)$, calculate dot-product and cosine similarity by hand, then check using the Python functions.
2. Why does the dot product change when every document weight is multiplied by 10, while cosine similarity does not?
3. A system retrieves 12 documents, 9 of which are relevant. If the collection contains 30 relevant documents, calculate recall and precision.

**Note:** The notebook retains the lecture's examples and introduces two clearly marked supplementary demonstrations.